In [10]:
import os
import pandas as pd
import shutil # <-- NEW: Import the shutil module for file operations

# === CONFIGURATION ===
# NOTE: Using raw string (r"...") is essential for Windows paths with backslashes
INPUT_FOLDER = fr"\\sanesstorage.cns.nyu.edu\archive\ginosar\Processed_data\Audio\272\Averaged_wavs_w_annotations_inspected"
OUTPUT_FOLDER = fr"\\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\vox\vox_new"
"/mnt/ceph/users/gginosar/"
"/mnt/home/neurostatslab/ceph/saneslab_data/big_setup/experiment_272/"
# Change this to your desired prefix (e.g., "exp_235_", "cleaned_A_")
FILE_PREFIX = "exp_272_"

# === MAIN LOOP ===
def clean_annotations(folder_INPUT, folder_OUTPUT):

    # 1. Ensure the output folder exists
    if not os.path.exists(folder_OUTPUT):
        os.makedirs(folder_OUTPUT)
        print(f"Created output directory: {folder_OUTPUT}")

    # 2. Loop through all files in the input folder
    for filename in os.listdir(folder_INPUT):
        input_path = os.path.join(folder_INPUT, filename)

        # Skip if it's a directory
        if os.path.isdir(input_path):
            continue

        # --- A. Handle .csv files (Existing logic) ---
        if filename.endswith(".csv"):
            # Construct the new output path with the custom prefix
            output_path = os.path.join(folder_OUTPUT, FILE_PREFIX + filename)

            try:
                df = pd.read_csv(input_path)

                # Skip if required columns are missing
                if not {'name', 'channel'}.issubset(df.columns):
                    print(f"⚠️ Skipping CSV {filename} (missing 'name' or 'channel')")
                    continue

                # Filter to only 'vox' rows
                df = df[df['name'] == 'vox'].copy()

                # Set all channels to -1
                df['channel'] = -1

                # Save cleaned file
                df.to_csv(output_path, index=False)
                print(f"✅ Saved cleaned CSV file: {output_path}")

            except Exception as e:
                print(f"❌ Error processing CSV {filename}: {e}")

        # --- B. Handle .wav files (NEW logic) ---
        elif filename.endswith(".wav"):
            # Construct the new output path with the custom prefix
            output_wav_filename = FILE_PREFIX + filename
            output_path = os.path.join(folder_OUTPUT, output_wav_filename)

            try:
                # Copy the file to the output folder and rename it
                shutil.copy2(input_path, output_path)
                print(f"🎶 Copied and renamed WAV file: {output_path}")

            except Exception as e:
                print(f"❌ Error copying WAV {filename}: {e}")

        else:
            print(f"Skipping non-CSV/non-WAV file: {filename}")


# === RUN ===
if __name__ == "__main__":
    clean_annotations(INPUT_FOLDER, OUTPUT_FOLDER)

Skipping non-CSV/non-WAV file: channel_10_file_005.npz
🎶 Copied and renamed WAV file: \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\vox\vox_new\exp_112_channel_10_file_005.wav
✅ Saved cleaned CSV file: \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\vox\vox_new\exp_112_channel_10_file_005_annotations.csv
Skipping non-CSV/non-WAV file: channel_10_file_014.npz
🎶 Copied and renamed WAV file: \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\vox\vox_new\exp_112_channel_10_file_014.wav
✅ Saved cleaned CSV file: \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\vox\vox_new\exp_112_channel_10_file_014_annotations.csv
Skipping non-CSV/non-WAV file: channel_10_file_021.npz
🎶 Copied and renamed WAV file: \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\vox\vox_new\exp_112_channel_10_file_021.wav
✅ Saved cleaned CSV file: \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\vox\vo

In [ ]:
import os
import numpy as np
from zipfile import BadZipFile

# === CONFIGURATION ===
# REPLACE THIS WITH THE FULL PATH TO THE DIRECTORY CONTAINING NPZ FILES
NPZ_DIRECTORY = fr"Z:\ginosar\Pre_processing\das\models\vox\vox_new"

# === MAIN FUNCTION ===
def find_corrupt_npz_files(directory_path):
    """
    Traverses a directory and checks all .npz files for integrity by attempting to load them.
    Reports files that fail the integrity check.
    """

    corrupt_files = []
    total_files_checked = 0

    print(f"--- Starting NPZ Integrity Check in: {directory_path} ---")

    # Use os.walk to search in subdirectories as well
    for root, _, files in os.walk(directory_path):
        for filename in files:
            if filename.endswith(".npz"):
                full_path = os.path.join(root, filename)
                total_files_checked += 1

                try:
                    # Attempt to load the file. This will raise an error if the file
                    # is not a valid zip archive or if it's truncated/corrupt.
                    with np.load(full_path, allow_pickle=False) as data:
                        # File loaded successfully. Do nothing here.
                        pass

                except BadZipFile:
                    # Catches errors specific to the file not being a valid zip structure
                    corrupt_files.append((full_path, "BadZipFile (Corrupted Archive)"))
                except (IOError, OSError, EOFError, ValueError) as e:
                    # Catches general I/O errors or errors from a truncated file
                    corrupt_files.append((full_path, f"I/O Error: {type(e).__name__}"))
                except Exception as e:
                    # Catches any other unexpected errors
                    corrupt_files.append((full_path, f"Unexpected Error: {type(e).__name__}"))


    # === SUMMARY REPORT ===
    print("\n--- INTEGRITY CHECK COMPLETE ---")
    print(f"Total .npz files checked: {total_files_checked}")

    if not corrupt_files:
        print("🎉 SUCCESS! All files loaded successfully. No corruption found.")
    else:
        print(f"⚠️ WARNING: Found {len(corrupt_files)} CORRUPT or unreadable file(s)!")

        for path, reason in corrupt_files:
            print(f"\n🚨 CORRUPT FILE FOUND 🚨")
            print(f"  Path: {path}")
            print(f"  Reason: {reason}")

        print("\nACTION REQUIRED: Please replace these files with fresh copies from the source.")


# === RUN ===
if __name__ == "__main__":
    find_corrupt_npz_files(NPZ_DIRECTORY)

--- Starting NPZ Integrity Check in: Z:\ginosar\Pre_processing\das\models\vox\vox_new ---

--- INTEGRITY CHECK COMPLETE ---
Total .npz files checked: 136
⚠️ WARNING: Found 1 CORRUPT or unreadable file(s)!

🚨 CORRUPT FILE FOUND 🚨
  Path: Z:\ginosar\Pre_processing\das\models\vox\vox_new\exp_112_channel_20_file_051.npz
  Reason: BadZipFile (Corrupted Archive)

ACTION REQUIRED: Please replace these files with fresh copies from the source.


In [11]:
import os
import pandas as pd
import shutil # Import the shutil module for file operations

# === CONFIGURATION ===
# NOTE: Using raw string (r"...") is essential for Windows paths with backslashes
INPUT_FOLDER = fr"\\sanesstorage.cns.nyu.edu\archive\ginosar\Processed_data\Audio\112\Averaged_wavs_w_annotations_inspected"
OUTPUT_FOLDER = fr"\\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\vox\vox_new"

FILE_PREFIX = "exp_112_"

# === MAIN LOOP ===
def clean_annotations(folder_INPUT, folder_OUTPUT):

    # 1. Ensure the output folder exists
    if not os.path.exists(folder_OUTPUT):
        os.makedirs(folder_OUTPUT)
        print(f"Created output directory: {folder_OUTPUT}")

    # 2. Loop through all files in the input folder
    for filename in os.listdir(folder_INPUT):
        input_path = os.path.join(folder_INPUT, filename)

        # Skip if it's a directory
        if os.path.isdir(input_path):
            continue

        # --- A. Handle .csv files (Cleaning and renaming) ---
        if filename.endswith(".csv"):
            # Construct the new output path with the custom prefix
            output_path = os.path.join(folder_OUTPUT, FILE_PREFIX + filename)

            try:
                df = pd.read_csv(input_path)

                # Skip if required columns are missing
                if not {'name', 'channel'}.issubset(df.columns):
                    print(f"⚠️ Skipping CSV {filename} (missing 'name' or 'channel')")
                    continue

                # Filter to only 'vox' rows
                df = df[df['name'] == 'vox'].copy()

                # Set all channels to -1
                df['channel'] = -1

                # Save cleaned file
                df.to_csv(output_path, index=False)
                print(f"✅ Saved cleaned CSV file: {output_path}")

            except Exception as e:
                print(f"❌ Error processing CSV {filename}: {e}")

        # --- B. Handle .wav files (Copying and renaming) ---
        elif filename.endswith(".wav"):
            # Construct the new output path with the custom prefix
            output_wav_filename = FILE_PREFIX + filename
            output_path = os.path.join(folder_OUTPUT, output_wav_filename)

            try:
                # Copy the file to the output folder and rename it
                shutil.copy2(input_path, output_path)
                print(f"🎶 Copied and renamed WAV file: {output_path}")

            except Exception as e:
                print(f"❌ Error copying WAV {filename}: {e}")

        # --- C. Handle .npz files (NEW: Copying and renaming) ---
        elif filename.endswith(".npz"):
            # Construct the new output path with the custom prefix
            output_npz_filename = FILE_PREFIX + filename
            output_path = os.path.join(folder_OUTPUT, output_npz_filename)

            try:
                # Copy the file to the output folder and rename it
                shutil.copy2(input_path, output_path)
                print(f"📦 Copied and renamed NPZ file: {output_path}")

            except Exception as e:
                print(f"❌ Error copying NPZ {filename}: {e}")

        else:
            print(f"Skipping unhandled file type: {filename}")


# === RUN ===
if __name__ == "__main__":
    clean_annotations(INPUT_FOLDER, OUTPUT_FOLDER)

📦 Copied and renamed NPZ file: \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\vox\vox_new\exp_112_channel_10_file_005.npz
🎶 Copied and renamed WAV file: \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\vox\vox_new\exp_112_channel_10_file_005.wav
✅ Saved cleaned CSV file: \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\vox\vox_new\exp_112_channel_10_file_005_annotations.csv
📦 Copied and renamed NPZ file: \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\vox\vox_new\exp_112_channel_10_file_014.npz
🎶 Copied and renamed WAV file: \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\vox\vox_new\exp_112_channel_10_file_014.wav
✅ Saved cleaned CSV file: \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\vox\vox_new\exp_112_channel_10_file_014_annotations.csv
📦 Copied and renamed NPZ file: \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\vox\vox_new\exp_112_ch

In [14]:
import os
import numpy as np

# === CONFIGURATION ===
# Replace this with the path to the directory containing your NPZ files
NPZ_DIRECTORY = fr"Z:\ginosar\Pre_processing\das\models\vox\vox_new"

# The key that is REQUIRED by the DAS tool
REQUIRED_KEY = 'samplerate'

# === MAIN FUNCTION ===
def check_npz_for_key(directory_path, required_key):
    """
    Traverses a directory and checks all .npz files for a specific required key.
    Prints a flag if the key is missing from any file.
    """

    # Counters for the summary report
    files_checked = 0
    missing_key_count = 0

    print(f"--- Starting NPZ Audit in: {directory_path} ---")
    print(f"Checking for required key: '{required_key}'\n")

    # Use os.walk to search in subdirectories as well
    for root, _, files in os.walk(directory_path):
        for filename in files:
            if filename.endswith(".npz"):
                full_path = os.path.join(root, filename)
                files_checked += 1

                try:
                    # Open the NPZ archive in read-only mode
                    with np.load(full_path, allow_pickle=True) as data:
                        # Check if the required key is in the archive's keys
                        if required_key not in data.keys():
                            print(f"🚨 MISSING KEY FLAG 🚨: '{required_key}' not found in:")
                            print(f"  -> File: {full_path}")
                            print(f"  -> Found Keys: {list(data.keys())}")
                            missing_key_count += 1

                except FileNotFoundError:
                    # This handles the intermittent network drive issue if the path is visible
                    # but not accessible at the moment of reading.
                    print(f"❌ ERROR: File not found (Access Issue?): {full_path}")
                except Exception as e:
                    print(f"❌ ERROR: Failed to read file {filename}. Reason: {e}")

    # === SUMMARY REPORT ===
    print("\n--- AUDIT COMPLETE ---")
    print(f"Total .npz files checked: {files_checked}")

    if missing_key_count == 0:
        print(f"🎉 SUCCESS! All files contain the required key '{required_key}'.")
    else:
        print(f"⚠️ WARNING: {missing_key_count} file(s) are missing the required key.")
        print("Please check the files flagged above.")


# === RUN ===
if __name__ == "__main__":
    check_npz_for_key(NPZ_DIRECTORY, REQUIRED_KEY)

--- Starting NPZ Audit in: Z:\ginosar\Pre_processing\das\models\vox\vox_new ---
Checking for required key: 'samplerate'

🚨 MISSING KEY FLAG 🚨: 'samplerate' not found in:
  -> File: Z:\ginosar\Pre_processing\das\models\vox\vox_new\exp_112_channel_20_file_034.npz
  -> Found Keys: ['data']
❌ ERROR: Failed to read file exp_112_channel_20_file_051.npz. Reason: File is not a zip file
🚨 MISSING KEY FLAG 🚨: 'samplerate' not found in:
  -> File: Z:\ginosar\Pre_processing\das\models\vox\vox_new\exp_112_channel_20_file_053.npz
  -> Found Keys: ['data']
🚨 MISSING KEY FLAG 🚨: 'samplerate' not found in:
  -> File: Z:\ginosar\Pre_processing\das\models\vox\vox_new\exp_112_channel_20_file_055.npz
  -> Found Keys: ['data']

--- AUDIT COMPLETE ---
Total .npz files checked: 139
⚠️ WARNING: 3 file(s) are missing the required key.
Please check the files flagged above.


In [3]:

import os
import re
import pandas as pd
from datetime import datetime, timedelta
from scipy.io import wavfile
import numpy as np
from tqdm import tqdm


In [4]:
exp = 276

folder_path_raw_wavs=fr"\\sanesstorage.cns.nyu.edu\archive\ginosar\Raw_data\experiment_{exp}\concatenated_data_cam_mic_sync\wavs"
folder_path_averaged_wavs = fr"\\sanesstorage.cns.nyu.edu\archive\ginosar\Processed_data\Audio\{exp}\Averaged_wavs_w_annotations"
folder_path_Audio = fr"\\sanesstorage.cns.nyu.edu\archive\ginosar\Processed_data\Audio\{exp}"


# --- Parameters ----------------------------------------------------
arena_wav_folder = folder_path_averaged_wavs  
sample_rate_expected = 125000
output_folder = folder_path_Audio  #  output path for CSVs

time_window_overlapping_calls_sec = 0.015
inter_call_interval_within_bout_threshold = 10.0  # seconds
min_calls_per_bout = 5 

# --------------------------------------------------------------------



# ##############################################################################
# load sync file
# ##############################################################################

def load_sync_file(exp):
    folder_path_sync = fr"\\sanesstorage.cns.nyu.edu\archive\ginosar\Raw_data\experiment_{exp}\concatenated_data_cam_mic_sync"
    sync_path = os.path.join(folder_path_sync, "sync.csv")
    sync_df = pd.read_csv(sync_path)
    sync_df['timestamp'] = sync_df['timestamp'].apply(eval)
    sync_df['start_time'] = sync_df['timestamp'].apply(lambda x: datetime.strptime(x[0], "%Y-%m-%d %H:%M:%S"))

    start_time_experiment = sync_df.loc[0,'start_time'] # Start of *experiment*
    print(f"Experiment {exp} started at: {start_time_experiment}")

    return start_time_experiment, sync_df

# Load sync file 
start_time_experiment,sync_df = load_sync_file(exp)
#sync_df.head()


Experiment 276 started at: 2025-07-08 16:17:54


In [5]:
# 0)------ Avergae audio data ----- > average two microphones for each arena
# ##############################################################################

import os
import re
import numpy as np
from scipy.io import wavfile

def average_microphone_pairs(exp, folder_path_raw_wavs, output_folder, channel_mapping, file_prefix="channel"):
    """
    Averages specified microphone channel pairs and saves them as new WAV files.

    channel_mapping={
        '10': [0, 1], arena 1
        '20': [2, 3], arena 2
        '30': [4, 5] underground
    Parameters:
        exp (int): Experiment number (used for logging).
        folder_path_raw_wavs (str): Path to the folder containing the raw WAV files.
        output_folder (str): Path to save the averaged WAV files.
        channel_mapping (dict): Dictionary mapping virtual channel IDs (e.g., '10') to pairs of real channels (e.g., [0, 1]).
        file_prefix (str): Prefix for the output filename, usually "channel" (default).
    """
    os.makedirs(output_folder, exist_ok=True)

    # Extract list of file numbers by scanning for one reference channel (e.g., channel_00)
    file_nums = sorted([
        int(re.match(rf"{file_prefix}_00_file_(\d+)\.wav", f).group(1)) # looking for the different file numebrs from channel 00, grouped by file number
        for f in os.listdir(folder_path_raw_wavs)
        if re.match(rf"{file_prefix}_00_file_(\d+)\.wav", f)
    ])

    print(f"Found {len(file_nums)} file chunks in experiment {exp}.")
    print(f"Saving in ",output_folder)

    for file_num in file_nums:
        file_num_str = f"{file_num:03d}"

        for virtual_ch, real_pair in channel_mapping.items():
            wav_data = []
            for ch in real_pair:
                fname = f"{file_prefix}_{ch:02d}_file_{file_num_str}.wav"
                fpath = os.path.join(folder_path_raw_wavs, fname)

                if not os.path.exists(fpath):
                    print(f" Missing file: {fname}")
                    break

                rate, data = wavfile.read(fpath)
                wav_data.append(data)

            # Check both files loaded successfully
            if len(wav_data) != 2:
                print(f" Skipping virtual channel {virtual_ch} file {file_num_str} (incomplete pair)")
                continue

            if wav_data[0].shape != wav_data[1].shape:
                print(f" Shape mismatch in file {file_num_str} for channels {real_pair}")
                continue

            avg_data = ((wav_data[0].astype(np.float32) + wav_data[1].astype(np.float32)) / 2).astype(np.float32)
            out_fname = f"{file_prefix}_{virtual_ch}_file_{file_num_str}.wav"
            out_fpath = os.path.join(output_folder, out_fname)

            wavfile.write(out_fpath, rate, avg_data)
            print(f"✅ Saved averaged file: {out_fname}")

    print(f"\n Finished averaging microphone pairs for experiment {exp}.")


# Averaging arena channels
average_microphone_pairs(
    exp,
    folder_path_raw_wavs=folder_path_raw_wavs,
    output_folder=folder_path_averaged_wavs,
    channel_mapping={
        '10': [2, 3],
        '20': [4, 5],
        '30': [0, 1]
    }
)
        # exp < 272
        # '10': [0, 1],
        # '20': [2, 3],
        # '30': [4, 5]


Found 143 file chunks in experiment 276.
Saving in  \\sanesstorage.cns.nyu.edu\archive\ginosar\Processed_data\Audio\276\Averaged_wavs_w_annotations
✅ Saved averaged file: channel_10_file_000.wav
✅ Saved averaged file: channel_20_file_000.wav
✅ Saved averaged file: channel_30_file_000.wav
✅ Saved averaged file: channel_10_file_001.wav
✅ Saved averaged file: channel_20_file_001.wav
✅ Saved averaged file: channel_30_file_001.wav
✅ Saved averaged file: channel_10_file_002.wav
✅ Saved averaged file: channel_20_file_002.wav
✅ Saved averaged file: channel_30_file_002.wav
✅ Saved averaged file: channel_10_file_003.wav
✅ Saved averaged file: channel_20_file_003.wav
✅ Saved averaged file: channel_30_file_003.wav
✅ Saved averaged file: channel_10_file_004.wav
✅ Saved averaged file: channel_20_file_004.wav
✅ Saved averaged file: channel_30_file_004.wav
✅ Saved averaged file: channel_10_file_005.wav
✅ Saved averaged file: channel_20_file_005.wav
✅ Saved averaged file: channel_30_file_005.wav
✅ Save

In [6]:
# ##############################################################################
#  Helper functions
# ##############################################################################
# ==================================================== FUNCTIONS ========================================================
# === Function: load sync file ===

# === Function: Names of files to read ===
def create_file_names_to_read(exp,channel_number,folder_path):
    pattern = re.compile(fr'^channel_{channel_number:02d}_file_(\d+).wav$')
    matching_files = [f for f in os.listdir(folder_path) if pattern.match(f)]
    matching_files.sort(key=lambda f: int(pattern.match(f).group(1)))
    print(f"Found {len(matching_files)} recording files")
    # for f in matching_files:
    #     print("Matched file:", f)
    return matching_files, pattern


# === Function: allow times to be in ms ===
def format_timedelta_ms(td):
            total_seconds = td.total_seconds()
            hours = int(total_seconds // 3600)
            minutes = int((total_seconds % 3600) // 60)
            seconds = total_seconds % 60
            return f"{hours:02}:{minutes:02}:{seconds:06.3f}"  # 3 decimal places = ms


# === Function: get playback times from TTL drops per file ===
column_order = [
    'file_num',
    'event_type',
    'channel',
    'start_time_file_sec',
    'stop_time_file_sec',
    'duration_sec',
    'start_time_real',
    'stop_time_real',
    'start_time_experiment',
    'stop_time_experiment',
    'start_time_experiment_sec',
    'stop_time_experiment_sec',
    'assigned_location'
]


In [7]:
# ##############################################################################
# 2. extract timings of playbacks (and robot - todo)
# ##############################################################################

from datetime import timedelta
import numpy as np

def extract_playback_times_per_file(df_ave_wav, sample_rate, start_time_file, start_time_experiment, file_number,channel_number_playback):
    threshold_TTL_flag = 0.5
    min_silence_duration_sec = 0.3

    # Skip files where TTL stream was never activated

    if df_ave_wav.max() < 1.0:
        return []

    
    # Skip the first part of the file until TTL stream is "active"
    activation_threshold = 1.5
    activation_window_sec = 0.2
    activation_window_samples = int(sample_rate * activation_window_sec)

    # Find first window where the TTL is active
    activated = df_ave_wav.rolling(activation_window_samples).mean() > activation_threshold
    first_activation_idx = activated[activated].first_valid_index()

    if first_activation_idx is None:
        return []

    # From here, work only with the post-activation segment
    df_ave_wav = df_ave_wav.iloc[first_activation_idx:].reset_index(drop=True)
        
    
 
    
    silence = df_ave_wav < threshold_TTL_flag
    diff = np.diff(silence.astype(int))
    flag_start_idxs = np.where(diff == 1)[0]
    flag_end_idxs = np.where(diff == -1)[0]

    # Align start and end indices
    if flag_end_idxs.shape[0] > 0 and flag_start_idxs.shape[0] > 0:
        if flag_end_idxs[0] < flag_start_idxs[0]:
            flag_end_idxs = flag_end_idxs[1:]
        if flag_start_idxs.shape[0] > flag_end_idxs.shape[0]:
            flag_start_idxs = flag_start_idxs[:flag_end_idxs.shape[0]]

    playback_info = []

    for start, end in zip(flag_start_idxs, flag_end_idxs): # define start/end a dip
        duration_sec = (end - start) / sample_rate
        if duration_sec < min_silence_duration_sec:
            continue  # skip short dips

        # Start time
        start_real = start_time_file + timedelta(seconds=start / sample_rate)
        start_rel_file_sec = round(start / sample_rate, 3)
        start_rel_exp_str = format_timedelta_ms(start_real - start_time_experiment)
        start_sec = (start_real - start_time_experiment).total_seconds()

        # End time
        stop_real = start_time_file + timedelta(seconds=end / sample_rate)
        end_rel_file_sec = round(end / sample_rate, 3)
        end_rel_exp_str = format_timedelta_ms(stop_real - start_time_experiment)
        stop_sec = (stop_real - start_time_experiment).total_seconds()

        # --- Speaker ID detection --- NOT WORKING
        speaker_id = None

        # Look 60 ms ahead, but skip the 10 ms high period
        lookahead_start = end + int(0.010 * sample_rate)
        lookahead_end = end + int(0.060 * sample_rate)
        speaker_window = silence[lookahead_start:lookahead_end]

        # Convert to numpy array
        speaker_window_np = speaker_window.to_numpy() if isinstance(speaker_window, pd.Series) else speaker_window

        # Find changes in silence pattern
        change_points = np.where(np.concatenate(([True], speaker_window_np[:-1] != speaker_window_np[1:], [True])))[0]
        silent_stretch_lengths = np.diff(change_points)

        assigned_location = 0
        # Determine speaker ID from the first silent stretch
        if len(silent_stretch_lengths) > 0 and speaker_window_np[0]:  # first stretch is silence
            first_silence_len = silent_stretch_lengths[0]

            # Use ±2 ms tolerance
            if 0.008 * sample_rate <= first_silence_len <= 0.012 * sample_rate:
                assigned_location = 1 # speaker ID
            elif 0.048 * sample_rate <= first_silence_len <= 0.052 * sample_rate:
                assigned_location = 2
        # ----------------------------------------------------

        playback_info.append({
            'event_type': 'playback',
            'channel': channel_number_playback,  
            'file_num': file_number,
            'start_time_file_sec': start_rel_file_sec,
            'stop_time_file_sec': end_rel_file_sec,
            'duration_sec': round(duration_sec, 3),
            'start_time_real': start_real,
            'stop_time_real': stop_real,         
            'start_time_experiment': start_rel_exp_str,
            'stop_time_experiment': end_rel_exp_str,
            'start_time_experiment_sec': start_sec,
            'stop_time_experiment_sec': stop_sec,
            'assigned_location': assigned_location

        })

    return playback_info




# === Function: extract PLAYBACK times all files 
def extract_playback_times_all_files(exp, folder_path_raw_wavs, column_order, sync_df, start_time_experiment):
    # find the correct playback channel
    # playback_channels_dict = { # dictionary experiment<>playback channel
    # 97: 7,
    # 98:7,
    # 99:7,
    # 109:7,
    # 110:7,
    # 111:7,
    # 112:7,
    # 113:7,
    # 114:7,
    # 115:7,
    # 116:7,
    # 117:7,
    # 211: 6,
    # 235: 6,
    # 237: 6,
    # 240: 5,
    # 241: 5,
    # 267: 7,
    # 271: 7,
    # 272: 7,
    #
    # }

    playback_channels_dict = {
    **dict.fromkeys([97,98,99,109,110,111,112,113,114,115,116,117,267,271], 7),
    **dict.fromkeys([211,235,237], 6),
    **dict.fromkeys([240,241], 5),
    **dict.fromkeys(range(272, 278), 7),   # 272..278 inclusive
}

    channel_number_playback = playback_channels_dict.get(exp)
    if channel_number_playback is None:
        channel_number_playback = 6 # usually the case when we have burrow

    # find all playback files
    all_playback_rows = []
    matching_files, pattern = create_file_names_to_read(exp,channel_number_playback,folder_path_raw_wavs)
    
    # ############ DBG ###########################################
    # look over only some of the files for DBG
    # max_files = 50
    # if max_files is not None:
    #     matching_files = matching_files[34:max_files]
    ##############################################################
    for file_name in matching_files: # loop through all playback files in exp
        file_number = int(pattern.match(file_name).group(1))
        file_path = os.path.join(folder_path_raw_wavs, file_name)
        print(file_path)
        
        # Find start real time for current file from sync file
        file_sync_row = sync_df.iloc[file_number]  # assuming row index matches file number
        
        try:
            file_sync_row = sync_df.iloc[file_number]
        except Exception as e:
            raise e
        start_time_file = file_sync_row['start_time']

        # Load WAV
        wav_sampling_rate, wav_data = wavfile.read(file_path)
        if wav_data.ndim > 1:  # stereo
            wav_data = wav_data[:, 0]
        assert wav_sampling_rate == 125000, f"Unexpected sample rate: {wav_sampling_rate}"

        # transofrm wav file to pd.Series 
        df_wav = pd.Series(wav_data) 
        averaging_window_samples = int(0.1 * wav_sampling_rate) # Number of samples present in the window
        df_ave_wav = df_wav.rolling(averaging_window_samples, center=True).mean()    # Getting the moving average 
        df_ave_wav.fillna(0)

        # Get playback timestamps
        playbacks = extract_playback_times_per_file(df_ave_wav, wav_sampling_rate, start_time_file, start_time_experiment,file_number,channel_number_playback)

        if not playbacks:
            #print(f" No valid playbacks found for file {file_number}, skipping")
            continue

        for pb in playbacks:
            
            all_playback_rows.append({
                'file_num': file_number,
                **pb
            })
    
    playback_df = pd.DataFrame(all_playback_rows)
    
    # Ensure DataFrame has correct structure even if empty
    if playback_df.empty:
        playback_df = pd.DataFrame(columns=column_order)
    else:
        #playback_df = playback_df[column_order]
        existing_columns = [col for col in column_order if col in playback_df.columns]
        playback_df = playback_df[existing_columns]


    output_path = fr"\\sanesstorage.cns.nyu.edu\archive\ginosar\Processed_data\Audio\{exp}\playback_times.csv"
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    playback_df.to_csv(output_path, index=False)     
    #print(playback_df.head(15))


# Load sync file 
start_time_experiment,sync_df = load_sync_file(exp)
sync_df = sync_df.reset_index(drop=True)

# Extract playback times 
extract_playback_times_all_files(exp, folder_path_raw_wavs, column_order, sync_df, start_time_experiment) 


Experiment 276 started at: 2025-07-08 16:17:54
Found 143 recording files
\\sanesstorage.cns.nyu.edu\archive\ginosar\Raw_data\experiment_276\concatenated_data_cam_mic_sync\wavs\channel_07_file_000.wav
\\sanesstorage.cns.nyu.edu\archive\ginosar\Raw_data\experiment_276\concatenated_data_cam_mic_sync\wavs\channel_07_file_001.wav
\\sanesstorage.cns.nyu.edu\archive\ginosar\Raw_data\experiment_276\concatenated_data_cam_mic_sync\wavs\channel_07_file_002.wav
\\sanesstorage.cns.nyu.edu\archive\ginosar\Raw_data\experiment_276\concatenated_data_cam_mic_sync\wavs\channel_07_file_003.wav
\\sanesstorage.cns.nyu.edu\archive\ginosar\Raw_data\experiment_276\concatenated_data_cam_mic_sync\wavs\channel_07_file_004.wav
\\sanesstorage.cns.nyu.edu\archive\ginosar\Raw_data\experiment_276\concatenated_data_cam_mic_sync\wavs\channel_07_file_005.wav
\\sanesstorage.cns.nyu.edu\archive\ginosar\Raw_data\experiment_276\concatenated_data_cam_mic_sync\wavs\channel_07_file_006.wav
\\sanesstorage.cns.nyu.edu\archive\gin

In [20]:
# ##############################################################################
# 3) extract timings of VOX
# ##############################################################################

# --- Function: extract vox times from DAS ---
print(folder_path_averaged_wavs)
def extract_vocalizations_all_files(exp, folder_path_averaged_wavs, sync_df, start_time_experiment,column_order,target_call_type=['alarm']):
    import glob

    rows = []

    # Pattern: channel_05_file_010.csv
    file_pattern = os.path.join(folder_path_averaged_wavs, "*.csv")
    all_files = glob.glob(file_pattern)

    for file_path in all_files:
        fname = os.path.basename(file_path)
        print(fname)
        match = re.match(r".*?_(\d+)_file_(\d+)_annotations\.csv", fname)
        if not match:
            continue

        channel = int(match.group(1))
        file_num = int(match.group(2))

        try:
            file_sync_row = sync_df.iloc[file_num]
            file_start_time = file_sync_row['start_time']
        except IndexError:
            print(f"⚠️ Sync data missing for file {file_num}")
            continue

        df = pd.read_csv(file_path)

        if 'name' not in df or 'start_seconds' not in df or 'stop_seconds' not in df:
            print(f"⚠️ File {fname} is missing expected columns.")
            continue

        # filter only vox (for now)
        #df_vox = df[df['name'] == target_call_type]
        df_vox = df[df['name'].isin(target_call_type)]

        # if df_vox.empty:
        #     print(f"📭 No 'vox' calls in {fname}")
        #     continue

        for _, row in df_vox.iterrows():
            start_file_sec = row['start_seconds']
            stop_file_sec = row['stop_seconds']

            # Skip rows with missing or non-positive start/stop times
            if pd.isna(start_file_sec) or pd.isna(stop_file_sec) or start_file_sec == 0 or stop_file_sec == 0:
                print(f"⚠️ Skipping malformed row in {fname}: start={start_file_sec}, stop={stop_file_sec}")
                continue

            # Convert to real and experiment time
            start_real = file_start_time + timedelta(seconds=start_file_sec)
            stop_real = file_start_time + timedelta(seconds=stop_file_sec)
            start_exp = format_timedelta_ms(start_real - start_time_experiment)
            stop_exp = format_timedelta_ms(stop_real - start_time_experiment)
            start_sec = (start_real - start_time_experiment).total_seconds()
            stop_sec = (stop_real - start_time_experiment).total_seconds()
            call_type = row['name']  # this will be 'vox' for now
            duration_sec = round(stop_file_sec - start_file_sec, 3)

            rows.append({
                'event_type': call_type,
                'channel': channel,
                'file_num': file_num,
                'start_time_file_sec': round(start_file_sec, 3),
                'stop_time_file_sec': round(stop_file_sec, 3),
                'duration_sec': duration_sec,
                'start_time_real': start_real,
                'stop_time_real': stop_real,
                'start_time_experiment': start_exp,
                'stop_time_experiment': stop_exp,
                'start_time_experiment_sec': start_sec,
                'stop_time_experiment_sec': stop_sec,
                'assigned_location':np.nan

            })

    vox_df = pd.DataFrame(rows)
    # If no rows, create empty DataFrame with correct columns
    if vox_df.empty:
        vox_df = pd.DataFrame(columns=column_order)
    else:
        for col in column_order:
            if col not in vox_df.columns:
                vox_df[col] = pd.NA
        vox_df = vox_df[column_order]



    output_path = fr"\\sanesstorage.cns.nyu.edu\archive\ginosar\Processed_data\Audio\{exp}\alarms\\vocalization_times.csv"
    vox_df.to_csv(output_path, index=False)
    print(f"✅ Saved {len(vox_df)} vocalizations to {output_path}")

    return vox_df

# Get all vox times
vox_df = extract_vocalizations_all_files(exp, folder_path_averaged_wavs, sync_df, start_time_experiment,column_order,target_call_type=['alarm']) # here we need annotations


\\sanesstorage.cns.nyu.edu\archive\ginosar\Processed_data\Audio\273\alarms\Averaged_wavs_w_annotations
channel_10_file_000_annotations.csv
channel_20_file_000_annotations.csv
channel_30_file_000_annotations.csv
channel_10_file_001_annotations.csv
channel_20_file_001_annotations.csv
channel_30_file_001_annotations.csv
channel_10_file_002_annotations.csv
channel_20_file_002_annotations.csv
channel_30_file_002_annotations.csv
channel_10_file_003_annotations.csv
channel_20_file_003_annotations.csv
channel_30_file_003_annotations.csv
channel_10_file_004_annotations.csv
channel_20_file_004_annotations.csv
channel_30_file_004_annotations.csv
channel_10_file_005_annotations.csv
channel_20_file_005_annotations.csv
channel_30_file_005_annotations.csv
channel_10_file_006_annotations.csv
channel_20_file_006_annotations.csv
channel_30_file_006_annotations.csv
channel_10_file_007_annotations.csv
channel_20_file_007_annotations.csv
⚠️ Skipping malformed row in channel_20_file_007_annotations.csv: sta

In [21]:
# Step 1 - extract RMS - each call from each arena



import os
import numpy as np
import pandas as pd
from scipy.io import wavfile


sources = {
    'arena_1': 'channel_10',
    'arena_2': 'channel_20',
    'underground': 'channel_30'
}

def compute_rms(signal, eps=1e-12):
    return 20 * np.log10(np.sqrt(np.mean(signal.astype(np.float32) ** 2)) + eps)

vox_df = pd.read_csv(fr"\\sanesstorage.cns.nyu.edu\archive\ginosar\Processed_data\Audio\{exp}\alarms\vocalization_times.csv")

# ===  Compute RMS for each vocalization per channel ===
for loc, ch_name in sources.items():
    rms_col = f'RMS_{loc}'
    rms_vals = []

    for _, row in tqdm(vox_df.iterrows(), total=len(vox_df), desc=f"RMS for {loc}"):
        file_str = f"{int(row['file_num']):03d}"
        wav_path = os.path.join(arena_wav_folder, f"{ch_name}_file_{file_str}.wav")

        try:
            rate, data = wavfile.read(wav_path)
            if data.ndim > 1:
                data = data[:, 0]
            if rate != sample_rate_expected:
                rms_vals.append(np.nan)
                continue
            start = int(row['start_time_file_sec'] * rate)
            stop = int(row['stop_time_file_sec'] * rate)
            segment = data[start:stop]
            rms_vals.append(compute_rms(segment))
        except:
            rms_vals.append(np.nan)

    vox_df[rms_col] = rms_vals

print("Finished RMS computation, attempting to save...")

vox_df.to_csv(os.path.join(output_folder, "step1_with_rms.csv"), index=False)
print(f"✅ Step 1 complete: saved with RMS values in ",output_folder)




RMS for underground: 100%|██████████| 6407/6407 [16:07<00:00,  6.63it/s] 


Finished RMS computation, attempting to save...
✅ Step 1 complete: saved with RMS values in  \\sanesstorage.cns.nyu.edu\archive\ginosar\Processed_data\Audio\273\alarms


In [22]:
# Step 2 - merge overlapping calls in different arenas
# Sort the vocalizations by file number and start time 

import pandas as pd
import numpy as np
from tqdm import tqdm
import os

# Load only the CSV from Step 1
step1_csv_path = os.path.join(folder_path_Audio, "step1_with_rms.csv")
vox_df = pd.read_csv(step1_csv_path)

# Automatically detect RMS columns by looking for column names starting with 'RMS_'
rms_cols = [col for col in vox_df.columns if col.startswith('RMS_')]

# Sort by file number and start time
vox_df = vox_df.sort_values(['file_num', 'start_time_file_sec']).reset_index(drop=True)

merged = []
merged_log_rows = []
used_indices = set()

# Iterate through all rows to group overlapping vocalizations
for i, row in tqdm(vox_df.iterrows(), total=len(vox_df), desc="Step 2: Merging overlapping vocalizations"):
    if i in used_indices:
        continue  # Skip rows already assigned to a group

    group = [row]
    used_indices.add(i)

    # Find other unused rows from the same file
    same_file = vox_df[vox_df['file_num'] == row['file_num']]
    same_file = same_file[~same_file.index.isin(used_indices)]

    # Group rows that start within the overlapping time window
    for j, other_row in same_file.iterrows():
        if abs(other_row['start_time_file_sec'] - row['start_time_file_sec']) <= time_window_overlapping_calls_sec:
            group.append(other_row)
            used_indices.add(j)

    group_df = pd.DataFrame(group)

    # If group is empty or all rows have missing RMS, skip
    if group_df.empty or group_df[rms_cols].isna().all(axis=1).all():
        continue

# choose best call by call with max rms from all arenas
    rms_sum = group_df[rms_cols].sum(axis=1)
    best_idx = rms_sum.idxmax()
    if best_idx not in group_df.index:
        continue

    best_row = group_df.loc[best_idx]
    merged.append(best_row)

    # === Log merged group ===
    if len(group_df) > 1:
        group_df = group_df.copy()
        group_df.insert(0, 'is_chosen', False)
        group_df.loc[best_idx, 'is_chosen'] = True

        merged_log_rows.append(group_df)
        merged_log_rows.append(pd.DataFrame([{}]))  # empty row separator

# final merged DataFrame
merged_df = pd.DataFrame(merged).reset_index(drop=True)
merged_df.to_csv(os.path.join(folder_path_Audio, "step2_merged.csv"), index=False)

# Merged log DataFrame (only if there were overlapping calls)
if merged_log_rows:
    merged_log_df = pd.concat(merged_log_rows, ignore_index=True)
    merged_log_df.to_csv(os.path.join(folder_path_Audio, "step2_merged_log.csv"), index=False)
    print("✅ Step 2 complete: Overlapping vocalizations merged and saved (with log).")
else:
    print("✅ Step 2 complete: No overlapping vocalizations to log.")




Step 2: Merging overlapping vocalizations: 100%|██████████| 6407/6407 [00:08<00:00, 737.96it/s]


✅ Step 2 complete: Overlapping vocalizations merged and saved (with log).


In [23]:
# step 3 - compute bouts


# --- Parameters ---
step2_csv_path = os.path.join(folder_path_Audio, "step2_merged.csv")  # Load Step 2 output
output_bouts_path = os.path.join(folder_path_Audio, "step3_with_bouts.csv")

# --- Load data ---
merged_df = pd.read_csv(step2_csv_path)

# Filter for alarm-type calls only
alarm_calls = merged_df[merged_df['event_type'] == 'alarm'].copy()

# Sort by file and time
alarm_calls = alarm_calls.sort_values(['file_num', 'start_time_experiment_sec']).reset_index(drop=True)

# Compute inter-call interval
alarm_calls['inter_call_interval_sec'] = alarm_calls['start_time_experiment_sec'].diff().shift(-1)

# Initialize bout_id and counter
alarm_calls['bout_id'] = pd.NA

bout_counter = 0
i = 0
# --- Identify bouts ---
while i < len(alarm_calls) - 1:
    temp_bout_indices = [i]
    j = i + 1
    while j < len(alarm_calls) and alarm_calls.loc[j - 1, 'inter_call_interval_sec'] < inter_call_interval_within_bout_threshold:
        temp_bout_indices.append(j)
        j += 1
    
    if len(temp_bout_indices) >= min_calls_per_bout:
        bout_counter += 1
        for idx in temp_bout_indices:
            alarm_calls.at[idx, 'bout_id'] = bout_counter
    
    i = j  # skip to the end of the potential bout, whether it was valid or not

# Add position within bout
alarm_calls['position_in_bout'] = alarm_calls.groupby('bout_id').cumcount() + 1
alarm_calls.loc[alarm_calls['bout_id'].isna(), 'position_in_bout'] = pd.NA

# Format columns
alarm_calls['bout_id'] = alarm_calls['bout_id'].astype('Int64')
alarm_calls['position_in_bout'] = alarm_calls['position_in_bout'].astype('Int64')

# --- Merge bout info back to all calls ---
merged_df = merged_df.drop(columns=['bout_id', 'position_in_bout'], errors='ignore')
merged_df = merged_df.merge(
    alarm_calls[['start_time_experiment_sec', 'file_num', 'bout_id', 'position_in_bout']],
    on=['start_time_experiment_sec', 'file_num'],
    how='left'
)

merged_df['exp'] = exp  # Add experiment number

# --- Save ---
merged_df.to_csv(output_bouts_path, index=False)
print("✅ Step 3 complete: bouts saved to", output_bouts_path)


✅ Step 3 complete: bouts saved to \\sanesstorage.cns.nyu.edu\archive\ginosar\Processed_data\Audio\273\alarms\step3_with_bouts.csv


In [24]:
# step 4 - choose highest RMS as source location of call

# --- Parameters ---
input_csv = os.path.join(folder_path_Audio, "step3_with_bouts.csv")
output_csv = os.path.join(folder_path_Audio, "step4_assigned_location.csv")

sources = ['arena_1', 'arena_2', 'underground']  # column suffixes

# --- Load Data ---
df = pd.read_csv(input_csv)

# --- Assign Location Based on Maximum RMS ---
def get_best_rms_location(row):
    rms_vals = {loc: row.get(f'RMS_{loc}', -np.inf) for loc in sources}
    return max(rms_vals, key=rms_vals.get)

df['assigned_location'] = df.apply(get_best_rms_location, axis=1)

# --- Save ---
df.to_csv(output_csv, index=False)
print(f"✅ Step 4 complete: source location assigned and saved to {output_csv}")


✅ Step 4 complete: source location assigned and saved to \\sanesstorage.cns.nyu.edu\archive\ginosar\Processed_data\Audio\273\alarms\step4_assigned_location.csv


In [55]:
# 4.5 files for Miles

# Input and output paths
input_csv = fr"\\sanesstorage.cns.nyu.edu\archive\ginosar\Processed_data\Audio\{exp}\step4_assigned_location.csv"
output_csv = input_csv.replace("step4_assigned_location", "vox_for_Miles")

# Read the CSV
df = pd.read_csv(input_csv)

# Define the mapping from location to channel
location_to_channel = {
    "underground": 30,
    "arena_1": 10,
    "arena_2": 20,
}

# Create the new column
df["assigned_channel"] = df["assigned_location"].map(location_to_channel)

# Select only desired columns
output_df = df[[
    "file_num",
    "event_type",
    "start_time_file_sec",
    "stop_time_file_sec",
    "duration_sec",
    "assigned_location",
    "assigned_channel"
]]

# Save to new CSV
output_df.to_csv(output_csv, index=False)
print(f"✅ Saved to: {output_csv}")


✅ Saved to: \\sanesstorage.cns.nyu.edu\archive\ginosar\Processed_data\Audio\235\vox_for_Miles.csv


In [22]:
from pathlib import Path
import pandas as pd
import re

# ----------------------------
# Config you can tweak
# ----------------------------
#EXPS = [235]  # order matters
BASE_AUDIO_DIR = Path(r"\\sanesstorage.cns.nyu.edu\archive\ginosar\Processed_data\Audio")
OUTPUT_CSV = BASE_AUDIO_DIR / "vox_for_Miles.csv"  # where to save the merged file

# Option A: set a single event type for *all* rows (e.g., "alarm")
EVENT_TYPE_OVERRIDE = None  # e.g., "alarm"

# Option B: map existing event types to new ones (only applied if OVERRIDE is None)
# Example: {"alarm_call": "alarm", "contact_call": "contact"}
EVENT_TYPE_MAP = {"alarm": "vox"}




# Channel mapping
LOCATION_TO_CHANNEL = {
    "underground": 30,
    "arena_1": 10,
    "arena_2": 20,
}

# Columns to keep in the final CSV (same as your original selection)
OUTPUT_COLUMNS = [
    "file_num",
    "event_type",
    "start_time_file_sec",
    "stop_time_file_sec",
    "duration_sec",
    "assigned_location",
    "assigned_channel",
]

# ----------------------------
# Helper to make file_num sortable even if it's zero-padded strings
# ----------------------------
def _coerce_file_num_to_int(series: pd.Series) -> pd.Series:
    """
    Tries to convert file_num to int. Works if values are ints or numeric strings.
    If not cleanly numeric, extracts trailing digits to sort by (e.g., 'file_00123' -> 123).
    Falls back to NaN where conversion fails.
    """
    try:
        return pd.to_numeric(series, errors="coerce")
    except Exception:
        pass

    # Fallback: extract last run of digits
    def extract_int(x):
        if pd.isna(x):
            return pd.NA
        m = re.search(r"(\d+)$", str(x))
        return int(m.group(1)) if m else pd.NA

    return series.map(extract_int)

# ----------------------------
# Main
# ----------------------------
all_parts = []
for exp in EXPS:
    input_csv = BASE_AUDIO_DIR / f"{exp}" / "alarms" / "step4_assigned_location_alarms.csv"
    df = pd.read_csv(input_csv)

    # Assigned channel
    df["assigned_channel"] = df["assigned_location"].map(LOCATION_TO_CHANNEL)

    # Event type handling
    if EVENT_TYPE_OVERRIDE is not None:
        df["event_type"] = EVENT_TYPE_OVERRIDE
    elif EVENT_TYPE_MAP is not None:
        df["event_type"] = df["event_type"].map(EVENT_TYPE_MAP).fillna(df["event_type"])

    # Sort within this experiment by file_num (if numeric-ish), but DO NOT reorder experiments.
    if "file_num" in df.columns:
        df["_file_num_sort"] = _coerce_file_num_to_int(df["file_num"])
        df = df.sort_values(by=["_file_num_sort", "start_time_file_sec"], na_position="last")
        df = df.drop(columns=["_file_num_sort"], errors="ignore")

    # Keep only desired columns (drop any missing gracefully)
    df = df[[c for c in OUTPUT_COLUMNS if c in df.columns]].copy()

    # Append in the *given* experiments order
    all_parts.append(df)

# Concatenate in the order of EXPS
out = pd.concat(all_parts, ignore_index=True)

# Ensure column order exactly as requested (if some were missing, they’ll be omitted)
out = out[[c for c in OUTPUT_COLUMNS if c in out.columns]]

# Save
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
out.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Combined {len(EXPS)} experiments into: {OUTPUT_CSV}")
print(f"   Rows total: {len(out)}")


✅ Combined 1 experiments into: \\sanesstorage.cns.nyu.edu\archive\ginosar\Processed_data\Audio\vox_for_Miles.csv
   Rows total: 692


In [80]:
# step 5 - combine vox and (if applicable) playbacks

# --- File paths ---
audio_path = os.path.join(folder_path_Audio, "step4_assigned_location.csv")  # from Step 4
playback_path = os.path.join(folder_path_Audio, "playback_times.csv")        # your playback file
output_path = os.path.join(folder_path_Audio, fr"all_events_{exp}.csv")

# --- Load data ---
audio_df = pd.read_csv(audio_path)

if os.path.exists(playback_path):
    playback_df = pd.read_csv(playback_path)

    # --- Match columns ---
    # Ensure all required columns exist in both, adding missing ones with NaN
    all_columns = set(audio_df.columns).union(set(playback_df.columns))
    for col in all_columns:
        if col not in audio_df.columns:
            audio_df[col] = pd.NA
        if col not in playback_df.columns:
            playback_df[col] = pd.NA

    # Reorder both to the same column order
    audio_df = audio_df[list(all_columns)]
    playback_df = playback_df[list(all_columns)]

    # --- Combine ---
    combined_df = pd.concat([audio_df, playback_df], ignore_index=True)
    combined_df = combined_df.sort_values(['file_num', 'start_time_file_sec']).reset_index(drop=True)
    combined_df['exp'] = exp 
else:
    combined_df = audio_df
    
# Define the desired column order
column_order = [
    'exp','file_num', 'event_type', 
    'start_time_file_sec', 'stop_time_file_sec',
    'start_time_experiment', 'stop_time_experiment',
    'start_time_experiment_sec', 'stop_time_experiment_sec',
    'start_time_real', 'stop_time_real', 
    'RMS_arena_1', 'RMS_arena_2', 'RMS_underground', 'channel', 'assigned_location',
    'duration_sec', 'bout_id', 'position_in_bout'
]

# Reorder columns (missing ones are ignored silently)
final_columns = [col for col in column_order if col in combined_df.columns]
combined_df = combined_df[final_columns + [col for col in combined_df.columns if col not in final_columns]]


# --- Save ---
combined_df.to_csv(output_path, index=False)
print(f"✅ Step 5 complete: combined vocalizations and playback saved to {output_path}")


✅ Step 5 complete: combined vocalizations and playback saved to \\sanesstorage.cns.nyu.edu\archive\ginosar\Processed_data\Audio\116\all_events_116.csv


C:\Users\gg3065\AppData\Local\Temp\ipykernel_2544\3927682897.py:28: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_df = pd.concat([audio_df, playback_df], ignore_index=True)


In [81]:

# step 6 - Create bout_summary csv for each exp before combining - for thump detection
import pandas as pd
import os

# === Input/Output ===
input_csv = fr"\\sanesstorage.cns.nyu.edu\archive\ginosar\Processed_data\Audio\{exp}\all_events_{exp}.csv"
output_csv = input_csv.replace("all_events", "bout_summary")

# === Load data ===
df = pd.read_csv(input_csv)
alarms_df = df[df["event_type"] == "alarm"].copy()

# === Tag each row with a unique bout identifier ===
alarms_df["safe_bout_id"] = alarms_df.apply(
    lambda row: f"bout_{int(row['bout_id'])}" if pd.notna(row["bout_id"]) else f"no_bout_row{row.name}",
    axis=1
)

# === Prepare summary rows (only include bouts with >1 calls) ===
summary_rows = []
seen_bouts = set()

for idx, row in alarms_df.iterrows():
    bout_key = row["safe_bout_id"]
    if bout_key in seen_bouts:
        continue
    seen_bouts.add(bout_key)

    group = alarms_df[alarms_df["safe_bout_id"] == bout_key]

    if len(group) <= 1:
        continue  # skip single-call bouts

    summary_rows.append({
        "exp": group["exp"].iloc[0],
        "file_index": group["file_num"].iloc[0],
        "channel": group["channel"].iloc[0],
        "bout_id": group["bout_id"].iloc[0],
        "bout_start_seconds": group["start_time_file_sec"].min(),
        "bout_end_seconds": group["stop_time_file_sec"].max(),
        "n_calls": len(group),
        "bout_duration_seconds": group["stop_time_experiment_sec"].max() - group["start_time_experiment_sec"].min()
    })

# === Save summary ===
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(output_csv, index=False)
print(f"✅ Saved to {output_csv}")


✅ Saved to \\sanesstorage.cns.nyu.edu\archive\ginosar\Processed_data\Audio\116\bout_summary_116.csv


In [13]:
# Add log - new

import os
import pandas as pd
from datetime import datetime

# === Parameters ===
first_exp = 235
exp_list = [235, 237]  # All experiments in the shared file
exp_str = "_".join(str(e) for e in exp_list)

base_dir = r"\\sanesstorage.cns.nyu.edu\archive\ginosar"
raw_data_dir = os.path.join(base_dir, "Raw_data", f"experiment_{first_exp}")
processed_data_dir = os.path.join(base_dir, "Processed_data", "Audio")
main_csv_path = os.path.join(processed_data_dir, f"all_events_{exp_str}.csv")

# === Locate log file for the first experiment ===
log_file = next(f for f in os.listdir(raw_data_dir) if f.startswith(f"experiment_{first_exp}_log") and f.endswith(".txt"))
log_path = os.path.join(raw_data_dir, log_file)

# === Load all_events CSV ===
df = pd.read_csv(main_csv_path, parse_dates=["start_time_real", "stop_time_real"])

# === Load sync file for first experiment ===
def load_sync_file(exp):
    folder_path_sync = os.path.join(base_dir, "Raw_data", f"experiment_{exp}", "concatenated_data_cam_mic_sync")
    sync_path = os.path.join(folder_path_sync, "sync.csv")
    sync_df = pd.read_csv(sync_path)
    sync_df['timestamp'] = sync_df['timestamp'].apply(eval)
    sync_df['start_time'] = sync_df['timestamp'].apply(lambda x: datetime.strptime(x[0], "%Y-%m-%d %H:%M:%S"))
    return sync_df['start_time'][0]

start_time_experiment = load_sync_file(first_exp)

# === Parse the log file ===
log_rows = []
with open(log_path, 'r') as f:
    for line in f:
        if ">" in line and "_" in line:
            try:
                time_str, note = line.strip().split(">", 1)
                log_time = datetime.strptime(time_str.strip(), "%Y-%m-%d_%H-%M-%S")
                log_rows.append({
                    "exp": first_exp,
                    "event_type": "log",
                    "start_time_real": log_time,
                    "start_time_experiment_sec": (log_time - start_time_experiment).total_seconds(),
                    "note": note.strip()
                })
            except Exception as e:
                print(f"⚠️ Skipping malformed line: {line.strip()}")

log_df = pd.DataFrame(log_rows)

# === Merge with all_events ===
df_with_log = pd.concat([df, log_df], ignore_index=True)
df_with_log = df_with_log.sort_values("start_time_experiment_sec").reset_index(drop=True)

# === Save updated version ===
output_path = os.path.join(processed_data_dir, f"all_events_{exp_str}_with_log.csv")
df_with_log.to_csv(output_path, index=False)
print(f"✅ Saved: {output_path}")


StopIteration: 

In [238]:
# Adding log 

import os
import pandas as pd
from datetime import datetime

# === Parameters ===
base_dir = fr"\\sanesstorage.cns.nyu.edu\archive\ginosar"
raw_data_dir = os.path.join(base_dir, "Raw_data", f"experiment_{exp}")
processed_data_dir = os.path.join(base_dir, "Processed_data", "Audio", str(exp))
main_csv_path = os.path.join(processed_data_dir, f"all_events_{exp}.csv")
log_file = next(f for f in os.listdir(raw_data_dir) if f.startswith(f"experiment_{exp}_log") and f.endswith(".txt"))
log_path = os.path.join(raw_data_dir, log_file)

# === Load original all_events CSV ===
df = pd.read_csv(main_csv_path, parse_dates=["start_time_real", "stop_time_real"])

# === Get experiment start time from sync file ===
def load_sync_file(exp):
    folder_path_sync = os.path.join(raw_data_dir, "concatenated_data_cam_mic_sync")
    sync_path = os.path.join(folder_path_sync, "sync.csv")
    sync_df = pd.read_csv(sync_path)
    sync_df['timestamp'] = sync_df['timestamp'].apply(eval)
    sync_df['start_time'] = sync_df['timestamp'].apply(lambda x: datetime.strptime(x[0], "%Y-%m-%d %H:%M:%S"))
    return sync_df['start_time'][0]

start_time_experiment = load_sync_file(exp)

# === Parse the log file ===
log_rows = []
with open(log_path, 'r') as f:
    for line in f:
        if ">" in line and "_" in line:
            try:
                time_str, note = line.strip().split(">", 1)
                log_time = datetime.strptime(time_str.strip(), "%Y-%m-%d_%H-%M-%S")
                log_rows.append({
                    "exp": exp,
                    "event_type": "log",
                    "start_time_real": log_time,
                    "start_time_experiment_sec": (log_time - start_time_experiment).total_seconds(),
                    "note": note.strip()
                })
            except Exception as e:
                print(f"⚠️ Skipping malformed line: {line.strip()}")

log_df = pd.DataFrame(log_rows)

# === Merge with main events ===
df_with_log = pd.concat([df, log_df], ignore_index=True)
df_with_log = df_with_log.sort_values("start_time_experiment_sec").reset_index(drop=True)

# === Save the updated CSV ===
output_path = os.path.join(processed_data_dir, f"all_events_{exp}_with_log.csv")
df_with_log.to_csv(output_path, index=False)
print(f"✅ Saved: {output_path}")


StopIteration: 

In [25]:
# ============================
# Step 7 – Concatenate experiments to a single CSV
#   - Global time origin = start of `first_exp`
#   - Safer bout_id remap: per-exp unique IDs -> contiguous global IDs
# ============================

import os
import pandas as pd
import numpy as np
from datetime import datetime

# === Parameters ===
exp_list   = [272,273]   # experiments to concatenate
first_exp  = 272                                # actual start of multi-exp run
base_folder = r"\\sanesstorage.cns.nyu.edu\archive\ginosar\Processed_data\Audio"

# Output file name
exp_str = "_".join(str(e) for e in sorted(exp_list))
output_filename = f"events_combined_{exp_str}.csv"
output_path = os.path.join(base_folder, output_filename)

# === Helper: load the *real* start time for a given experiment from sync.csv ===
def load_sync_file(exp: int):
    sync_path = fr"\\sanesstorage.cns.nyu.edu\archive\ginosar\Raw_data\experiment_{exp}\concatenated_data_cam_mic_sync\sync.csv"
    if not os.path.exists(sync_path):
        raise FileNotFoundError(f"Sync file not found: {sync_path}")

    s = pd.read_csv(sync_path)

    # parse list-likes stored as strings (if needed)
    if isinstance(s["video"].iloc[0], str):
        s["video"] = s["video"].apply(eval)
    if isinstance(s["timestamp"].iloc[0], str):
        s["timestamp"] = s["timestamp"].apply(eval)

    # experiment start = first video's start timestamp
    exp_start_real = pd.to_datetime(s.iloc[0]["timestamp"][0])
    return exp_start_real, s

# === Global origin: start time of `first_exp` ===
true_experiment_start, _ = load_sync_file(first_exp)

# === Process and combine ===
dfs = []
global_bout_offset = 0  # how many unique bouts assigned so far globally

for exp in sorted(exp_list):
    csv_path = os.path.join(base_folder, str(exp), f"all_events_{exp}.csv")
    if not os.path.exists(csv_path):
        print(f"⚠️ File not found for exp {exp}: {csv_path}")
        continue

    df = pd.read_csv(csv_path)

    # Ensure exp column
    if "exp" not in df.columns:
        df["exp"] = exp
    else:
        df.loc[:, "exp"] = df["exp"].fillna(exp).astype(int)

    # Recompute global experiment-relative time (origin = start of first_exp)
    for col in ["start_time_real", "stop_time_real"]:
        df[col] = pd.to_datetime(df[col], errors="coerce")

    df["start_time_experiment_sec"] = (df["start_time_real"] - true_experiment_start).dt.total_seconds()
    df["stop_time_experiment_sec"]  = (df["stop_time_real"]  - true_experiment_start).dt.total_seconds()

    df["start_time_experiment"] = pd.to_timedelta(df["start_time_experiment_sec"], unit="s")
    df["stop_time_experiment"]  = pd.to_timedelta(df["stop_time_experiment_sec"],  unit="s")

    # --- SAFER bout_id remap (contiguous global IDs) ---
    if "bout_id" in df.columns:
        # Coerce to numeric; keep NaN for non-bout rows
        df["bout_id"] = pd.to_numeric(df["bout_id"], errors="coerce")

        # Unique *valid* (non-NaN) bout ids for this exp
        unique_local = df.loc[df["bout_id"].notna(), "bout_id"].astype(int).unique()
        if len(unique_local) > 0:
            unique_local = np.sort(unique_local)
            # map local IDs -> global contiguous IDs above current offset
            mapping = {old: (i + 1 + global_bout_offset) for i, old in enumerate(unique_local)}
            mask = df["bout_id"].notna()
            df.loc[mask, "bout_id"] = df.loc[mask, "bout_id"].astype(int).map(mapping)
            # advance global offset by the number of unique bouts we just assigned
            global_bout_offset += len(unique_local)

    dfs.append(df)

# === Save output ===
if dfs:
    combined_df = pd.concat(dfs, ignore_index=True)

    # (Optional) enforce integer dtype for bout_id (nullable Int64) so NaNs are allowed
    if "bout_id" in combined_df.columns:
        combined_df["bout_id"] = pd.to_numeric(combined_df["bout_id"], errors="coerce").astype("Int64")

    combined_df.to_csv(output_path, index=False)
    print(f"✅ Combined CSV saved to: {output_path}")

    # === Quick sanity checks ===
    chk = combined_df[combined_df["event_type"] == "alarm"].copy()
    if not chk.empty:
        chk["bout_id"] = pd.to_numeric(chk["bout_id"], errors="coerce")
        print("\nAlarm rows with non-null bout_id per exp:")
        print(chk.groupby("exp")["bout_id"].apply(lambda s: s.notna().sum()))
        print("\nUnique bout IDs per exp:")
        print(chk.groupby("exp")["bout_id"].nunique(dropna=True))

else:
    print("❌ No valid files were found.")


✅ Combined CSV saved to: \\sanesstorage.cns.nyu.edu\archive\ginosar\Processed_data\Audio\events_combined_272_273.csv

Alarm rows with non-null bout_id per exp:
exp
272    1697
273    3963
Name: bout_id, dtype: int64

Unique bout IDs per exp:
exp
272    159
273    378
Name: bout_id, dtype: int64


In [69]:
# OLD - rename all files of older experiments
import os
import re
import shutil

# === CONFIG ===
exp = 116  # 
old_folder = fr"\\sanesstorage.cns.nyu.edu\archive\ginosar\Raw_data\experiment_{exp}\concatenated_data_cam_mic_sync"
new_folder = fr"\\sanesstorage.cns.nyu.edu\archive\ginosar\Raw_data\experiment_{exp}\concatenated_data_cam_mic_sync\wavs"

os.makedirs(new_folder, exist_ok=True)

# === Match old filenames like: channel_3_10.wav or channel_3.010.wav
pattern = re.compile(r"channel[_\.](\d+)[_\.](\d+)\.wav")
#pattern = re.compile(r"channel[_\.](0?[5-7])[_\.](\d+)\.wav")


renamed_files = 0

for fname in os.listdir(old_folder):
    match = pattern.match(fname)
    if match:
        ch_num = int(match.group(1))
        file_num = int(match.group(2))

        # Format as channel_03_file_010.wav
        new_fname = f"channel_{ch_num:02d}_file_{file_num:03d}.wav"

        old_path = os.path.join(old_folder, fname)
        new_path = os.path.join(new_folder, new_fname)

        shutil.copy2(old_path, new_path)
        renamed_files += 1

        print(f"✔ Renamed: {fname} → {new_fname}")

print(f"\n✅ Completed: {renamed_files} files copied to '{new_folder}'")


✔ Renamed: channel_0_0.wav → channel_00_file_000.wav
✔ Renamed: channel_0_1.wav → channel_00_file_001.wav
✔ Renamed: channel_0_2.wav → channel_00_file_002.wav
✔ Renamed: channel_0_3.wav → channel_00_file_003.wav
✔ Renamed: channel_0_4.wav → channel_00_file_004.wav
✔ Renamed: channel_0_5.wav → channel_00_file_005.wav
✔ Renamed: channel_1_0.wav → channel_01_file_000.wav
✔ Renamed: channel_1_1.wav → channel_01_file_001.wav
✔ Renamed: channel_1_2.wav → channel_01_file_002.wav
✔ Renamed: channel_1_3.wav → channel_01_file_003.wav
✔ Renamed: channel_1_4.wav → channel_01_file_004.wav
✔ Renamed: channel_1_5.wav → channel_01_file_005.wav
✔ Renamed: channel_2_0.wav → channel_02_file_000.wav
✔ Renamed: channel_2_1.wav → channel_02_file_001.wav
✔ Renamed: channel_2_2.wav → channel_02_file_002.wav
✔ Renamed: channel_2_3.wav → channel_02_file_003.wav
✔ Renamed: channel_2_4.wav → channel_02_file_004.wav
✔ Renamed: channel_2_5.wav → channel_02_file_005.wav
✔ Renamed: channel_3_0.wav → channel_03_file_0

In [12]:
import os
import pandas as pd

# === CONFIGURATION ===
INPUT_FOLDER = r"\\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\vox\TRAIN_vox_112_old"
folder_path_out = r"\\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\vox\TRAIN_vox_112"

# === MAIN LOOP ===
def clean_annotations(folder_path):
    for filename in os.listdir(folder_path):
        if filename.endswith(".csv"):
            input_path = os.path.join(folder_path, filename)
            output_path = os.path.join(folder_path_out, filename)

            try:
                df = pd.read_csv(input_path)

                # Skip if required columns are missing
                if not {'name', 'channel'}.issubset(df.columns):
                    print(f"⚠️ Skipping {filename} (missing 'name' or 'channel')")
                    continue

                # Filter to only 'vox' rows
                df = df[df['name'] == 'vox'].copy()

                # Set all channels to -1
                df['channel'] = -1

                # Save cleaned file
                df.to_csv(output_path, index=False)
                print(f"✅ Saved cleaned file: {output_path}")

            except Exception as e:
                print(f"❌ Error processing {filename}: {e}")

# === RUN ===
if __name__ == "__main__":
    clean_annotations(INPUT_FOLDER)


✅ Saved cleaned file: \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\vox\TRAIN_vox_112\channel_10_file_005_annotations.csv
✅ Saved cleaned file: \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\vox\TRAIN_vox_112\channel_10_file_014_annotations.csv
✅ Saved cleaned file: \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\vox\TRAIN_vox_112\channel_10_file_021_annotations.csv
✅ Saved cleaned file: \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\vox\TRAIN_vox_112\channel_10_file_028_annotations.csv
✅ Saved cleaned file: \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\vox\TRAIN_vox_112\channel_10_file_033_annotations.csv
✅ Saved cleaned file: \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\vox\TRAIN_vox_112\channel_10_file_046_annotations.csv
✅ Saved cleaned file: \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\vox\TRAIN_vox_112\channel_10_file_101_a